# ✨ LivePortrait Studio - Lái Chuyển Động Cho Ảnh Nhân Vật (Tự Động Lưu Vào Google Drive)
> **Hướng dẫn 1-Click (Không cần tải lại lần sau):**
> 1. Bấm nút **'Sao chép vào Drive'** ở thanh trên cùng để lưu vĩnh viễn notebook này vào Google Drive của bạn.
> 2. Bấm **Thời gian chạy (Runtime)** -> **Thay đổi loại phần cứng** -> Chọn **T4 GPU**.
> 3. Bấm nút ▶️ ở ô lệnh bên dưới -> Bấm **'Kết nối với Google Drive'** khi được hỏi.
> 4. Lần đầu sẽ tải trọng số thật và lưu vào Drive, **từ lần thứ 2 trở đi sẽ nạp tức thì từ Drive mà KHÔNG CẦN TẢI LẠI!**

In [ ]:
#@title 🚀 Khởi chạy LivePortrait WebUI 1-Click (Tự Động Caching Google Drive)
import os
import sys
import time
import shutil
import subprocess
from google.colab import drive

# 1. Gắn kết Google Drive thông minh
print("🔗 [1/6] Đang kiểm tra kết nối Google Drive...")
has_drive = False
if os.path.exists('/content/drive/MyDrive'):
    print("🎉 Google Drive đã gắn kết sẵn!")
    has_drive = True
else:
    try:
        drive.mount('/content/drive')
        has_drive = True
    except Exception as e:
        print(f"⚠️ Không thể kết nối Drive tự động ({e}). Sẽ chạy trên bộ nhớ tạm của Colab.")

drive_cache_dir = "/content/drive/MyDrive/AI_Colab_Cache/LivePortrait" if has_drive else "/content/cache/LivePortrait"
drive_weights_dir = f"{drive_cache_dir}/pretrained_weights"
os.makedirs(drive_cache_dir, exist_ok=True)

# 2. Tải mã nguồn LivePortrait
print("⚡ [2/6] Đang chuẩn bị mã nguồn LivePortrait...")
%cd /content
if not os.path.exists("/content/LivePortrait"):
    !git clone -b dev https://github.com/camenduru/LivePortrait /content/LivePortrait

%cd /content/LivePortrait

# 3. Cài đặt và xác nhận từng thư viện bắt buộc (Dừng ngay nếu có lỗi, không chạy tiếp)
print("📦 [3/6] Đang kiểm tra và cài đặt thư viện môi trường...")
def ensure_pkg(pkg_install_cmd, import_name):
    try:
        __import__(import_name)
        print(f"  ✓ {import_name}: Sẵn sàng")
    except ImportError:
        print(f"  ⏳ Đang cài đặt {pkg_install_cmd}...")
        res = subprocess.run([sys.executable, "-m", "pip", "install", pkg_install_cmd], capture_output=True, text=True)
        if res.returncode != 0:
            print(f"  ❌ Lỗi khi cài đặt {pkg_install_cmd}:\n{res.stderr}")
            raise RuntimeError(f"Cài đặt {pkg_install_cmd} thất bại!")
        __import__(import_name)
        print(f"  ✓ {import_name}: Cài đặt thành công")

ensure_pkg("tyro", "tyro")
ensure_pkg("gradio", "gradio")
ensure_pkg("onnx", "onnx")
ensure_pkg("onnxruntime-gpu", "onnxruntime")
ensure_pkg("colorama", "colorama")
ensure_pkg("ffmpeg-python", "ffmpeg")
ensure_pkg("huggingface_hub", "huggingface_hub")

# 4. Vá lỗi PyTorch 2.6 toàn cục qua sitecustomize.py
print("🔧 [4/6] Cấu hình vá lỗi PyTorch 2.6 toàn cục...")
import site
for p in site.getsitepackages():
    sc_file = os.path.join(p, 'sitecustomize.py')
    with open(sc_file, 'w', encoding='utf-8') as f:
        f.write('''import torch
_old_torch_load = torch.load
def _safe_torch_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return _old_torch_load(*args, **kwargs)
torch.load = _safe_torch_load
''')
    break

!git checkout src/utils/helper.py 2>/dev/null || true

# 5. Kiểm tra toàn bộ 8 file trọng số cốt lõi
print("💾 [5/6] Kiểm tra bộ trọng số AI...")
core_files = [
    "liveportrait/base_models/appearance_feature_extractor.pth",
    "liveportrait/base_models/motion_extractor.pth",
    "liveportrait/base_models/spade_generator.pth",
    "liveportrait/base_models/warping_module.pth",
    "liveportrait/retargeting_models/stitching_retargeting_module.pth",
    "liveportrait/landmark.onnx",
    "insightface/models/buffalo_l/det_10g.onnx",
    "insightface/models/buffalo_l/2d106det.onnx"
]

all_drive_weights_valid = True
for f in core_files:
    target = os.path.join(drive_weights_dir, f)
    if not os.path.exists(target) or os.path.getsize(target) < 100000:
        all_drive_weights_valid = False
        break

if all_drive_weights_valid:
    print("  🎉 ĐÃ TÌM THẤY ĐỦ TOÀN BỘ 8 FILE TRỌNG SỐ THẬT TRONG GOOGLE DRIVE! Nạp trực tiếp trong 3 giây...")
    !rm -rf /content/LivePortrait/pretrained_weights
    !cp -r "{drive_weights_dir}" /content/LivePortrait/pretrained_weights
else:
    print("  ⏳ Đang tải toàn bộ trọng số AI thật (~650MB) từ HuggingFace...")
    !rm -rf /content/LivePortrait/pretrained_weights "{drive_weights_dir}"
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id='camenduru/LivePortrait', local_dir='/content/LivePortrait/pretrained_weights', local_dir_use_symlinks=False)
    if has_drive:
        print("  💾 Đang lưu bản sao trọng số thật vào Google Drive để lần sau nạp ngay lập tức...")
        os.makedirs(drive_cache_dir, exist_ok=True)
        !cp -r /content/LivePortrait/pretrained_weights "{drive_cache_dir}/"

# 6. Biên dịch module Cython 3D
print("⚙️ [6/6] Biên dịch Cython và khởi động WebUI...")
%cd /content/LivePortrait/src/utils/dependencies/insightface/thirdparty/face3d/mesh/cython
!{sys.executable} setup.py build_ext --inplace

%cd /content/LivePortrait

# Khởi động Cloudflare Tunnel dự phòng
!curl -LOs https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb >/dev/null 2>&1
!nohup cloudflared tunnel --url http://127.0.0.1:8890 > /content/cloudflared.log 2>&1 &

time.sleep(4)
cf_link = ""
if os.path.exists("/content/cloudflared.log"):
    import re
    with open("/content/cloudflared.log", "r", encoding="utf-8", errors="ignore") as f:
        matches = re.findall(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", f.read())
        if matches:
            cf_link = matches[0]

with open('/content/LivePortrait/app.py', 'r', encoding='utf-8') as f:
    app_text = f.read()

if 'while True: time.sleep(3600)' not in app_text:
    app_text += '\nimport time\nwhile True:\n    time.sleep(3600)\n'
    with open('/content/LivePortrait/app.py', 'w', encoding='utf-8') as f:
        f.write(app_text)

print("=" * 65)
print("🚀 LIVEPORTRAIT WEBUI ĐANG KHỞI ĐỘNG...")
if cf_link:
    print(f"🔗 LINK CLOUDFLARE (TRUY CẬP NGAY): {cf_link}")
print("🔗 LINK GRADIO LIVE SẼ XUẤT HIỆN NGAY BÊN DƯỚI:")
print("=" * 65)

!{sys.executable} -u app.py --share --server_port 8890
